# Figure 4 — organ-specific mesenchyme

The translational payoff. IRIS predicts signaling in the mesenchymal fates of
the E9–E9.5 mouse foregut, and the prediction that **WNT and BMP are enriched
in respiratory mesenchyme** motivated two wet-lab results:

1. ex vivo foregut culture — WNT activation expands *Tbx4*/*Foxf1*; WNT
   inhibition abolishes *Tbx4* (Fig. 4b)
2. a revised hESC protocol — WNT earlier and for longer, raising TBX4
   (Fig. 4c–e)

Script equivalent: `figures/fig4/fig4a_mesenchyme_enrichment.py`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "iris_repro").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "figures"))

import numpy as np
import pandas as pd
from iris_repro import config, data, metrics, plotting, provenance

plt = plotting.set_style()
OUT = config.output_dir("fig4")
print("outputs ->", OUT)

In [ ]:
from fig4.fig4a_mesenchyme_enrichment import (_canonical, POPULATIONS, TARGET,
                                             load_predictions)
print("comparing:", TARGET, "vs", [p for p in POPULATIONS if p != TARGET])

## The four divergent mesenchymal fates

The source object bundles several atlases and annotates sub-states separately
(`respiratory-lung`, `esophagus-1`, …); `_canonical` collapses those onto the
four fates compared in Fig. 4a and drops everything else.

In [ ]:
import anndata as ad

path = config.data_path("foregut_mesenchyme")
adata = ad.read_h5ad(path, backed="r")
pops = pd.Series([_canonical(v) for v in adata.obs["celltype"].astype(str)])
print(f"{pops.notna().sum():,} mesenchymal cells of {len(pops):,} in the file")
pops.value_counts()

## Enrichment

One-sided Fisher's exact test, respiratory vs all other foregut mesenchyme.

Predictions must exist first; generate them once with:

```bash
python figures/fig4/fig4a_mesenchyme_enrichment.py --predict
```

In [ ]:
try:
    df = load_predictions()
    df["population"] = [_canonical(v) for v in df["population"]]
    df = df[df["population"].notna()].copy()
    have_predictions = True
    display(df["population"].value_counts())
except SystemExit as exc:
    have_predictions = False
    print(exc)

In [ ]:
if have_predictions:
    is_target = (df["population"] == TARGET).to_numpy()
    rows = []
    for sig in config.signals():
        active = df[f"{sig}_pred"].to_numpy().astype(bool)
        table = [[int((is_target & active).sum()), int((is_target & ~active).sum())],
                 [int((~is_target & active).sum()), int((~is_target & ~active).sum())]]
        r = metrics.fisher_enrichment(table, alternative="greater")
        rows.append({"signal": config.display_name(sig),
                     "respiratory_active_frac": round(df.loc[is_target, f"{sig}_pred"].mean(), 3),
                     "other_active_frac": round(df.loc[~is_target, f"{sig}_pred"].mean(), 3),
                     "odds_ratio": round(r["odds_ratio"], 2), "p": r["p"],
                     "significant": r["p"] < 0.05})
    display(pd.DataFrame(rows))
else:
    print("Run the script with --predict first.")

Expect WNT and BMP to come out significantly enriched — that is the
prediction the ex vivo experiment tested.

## Why this mattered

The published protocol applied WNT only in the last 24 h, to push trachea
fate *after* respiratory commitment. IRIS placed WNT **at** commitment, so the
revised protocol starts WNT at day 4 (splanchnic mesoderm, ~E8.5) and holds it
longer. An intermediate CHIR dose is optimal: too much WNT suppresses FOXF1
(Supp. Fig. 21b,c).

In [ ]:
protocol = pd.DataFrame([
    {"protocol": "original", "WNT window": "day 6-7 only (24 h)",
     "CHIR": "2 uM", "rationale": "trachea fate, after respiratory commitment"},
    {"protocol": "IRIS-optimized", "WNT window": "day 4-7 (extended)",
     "CHIR": "3 uM", "rationale": "WNT acts AT respiratory commitment"},
])
protocol

Quantified outcomes are in Fig. 4e and Supp. Fig. 21d,e (one-sided
Student's t-test per gene, three biological replicates, repeated twice).
TBX4 rises strongly in both hESC and iPSC lines; FOXF1 is preserved at the
intermediate dose.